# Lesson 13a: Alignment — RLHF and Preference Optimisation — Theory

11a and 12a pretrained and fine-tuned a language model to predict
likely continuations of text — but "likely" and "preferred by a human"
are not the same target. This lesson derives how a model gets pointed at
the second target instead of the first: learn a **reward model** from
human *pairwise preferences* (never a hand-written score), use it to
drive reinforcement learning against the language model (**RLHF**), and
then derive a way to skip the reward model and the RL loop entirely
(**Direct Preference Optimisation**, DPO) while optimising, provably,
the same underlying objective.

By the end of this notebook you will have:
- explained why preference **comparisons**, not hand-written scores, are
  what alignment data actually consists of,
- derived the **Bradley-Terry model** and its reward-model loss, and
  recovered a hidden true reward ranking from nothing but noisy pairwise
  comparisons,
- described the **RLHF loop** end to end and identified exactly which
  step is a reinforcement learning problem, and
- derived **DPO** from the same preference model and implemented it from
  scratch, verified against PyTorch autograd.

## Introduction

A pretrained model's objective (11a) is a purely statistical one —
predict the next token a large text corpus actually contains. Nothing
about that objective distinguishes a helpful, honest response from a
fluent but unhelpful or actively misleading one; both can be equally
*likely* continuations of a prompt seen during pretraining. Alignment is
the step that changes what the model is optimised *for*, not what it is
trained *on* — and doing that requires a way to define "better response"
that a gradient can actually be computed against.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (preference sampling, weight init)
# is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

## Learning from Preferences

Writing a reward function for "a good response" by hand runs into the
same problem 8a's distributional hypothesis was built to avoid for word
meaning: quality is multi-dimensional, context-dependent, and hard to
specify completely — and a reward that is even slightly wrong in its
specification gets *optimised against directly*, so any gap between
"what the reward measures" and "what humans actually want" gets
exploited rather than ignored (Goodhart's law, operating on a language
model). What humans *are* reliably good at is the much narrower judgment
"is response A better than response B?" — a comparison, not an absolute
score, and comparisons made independently by different people are far
more consistent with each other than absolute ratings tend to be.
Preference-based alignment starts from exactly that: collect pairwise
comparisons, and *learn* a scalar reward model consistent with them,
rather than writing the reward down directly.

## The Bradley-Terry Model

The Bradley-Terry model gives comparisons a precise probabilistic
meaning: each item $i$ has a latent scalar reward $r_i$, and the
probability that $i$ beats $j$ in a head-to-head comparison is a
logistic function of the reward *difference*:

$$P(i \succ j) = \sigma(r_i - r_j) = \frac{e^{r_i}}{e^{r_i} + e^{r_j}}.$$

Given a dataset of observed comparisons — for each pair, which one a
human actually preferred — the reward model $r_\theta$ is fit by
maximising the likelihood of the data under this model, equivalently
minimising the negative log-likelihood over every observed comparison
$(w, l)$ (winner, loser):

$$\mathcal{L}(\theta) = -\sum_{(w,l)} \log \sigma\big(r_\theta(w) - r_\theta(l)\big).$$

This is exactly a logistic-regression loss on reward *differences* —
nothing more exotic is needed to turn "which one did the human prefer"
into a trainable scalar reward.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


# A hidden true reward per item, never revealed to the reward model --
# only noisy pairwise comparisons sampled from the Bradley-Terry model
# itself are observed, exactly as a real preference dataset would be.
items = ["A", "B", "C", "D", "E", "F"]
n_items = len(items)
true_reward = np.array([2.0, 1.2, 0.4, -0.3, -1.0, -1.8])

rng = np.random.default_rng(SEED)
n_comparisons = 400
pairs = rng.integers(0, n_items, size=(n_comparisons, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
win_probs = sigmoid(true_reward[pairs[:, 0]] - true_reward[pairs[:, 1]])
first_wins = rng.random(len(pairs)) < win_probs
winners = np.where(first_wins, pairs[:, 0], pairs[:, 1])
losers = np.where(first_wins, pairs[:, 1], pairs[:, 0])
print(f"{len(pairs)} pairwise comparisons sampled from the hidden true rewards")

# Fit r_hat by gradient descent on the Bradley-Terry negative log-likelihood.
r_hat = np.zeros(n_items)
lr, n_epochs = 0.1, 200
for epoch in range(n_epochs):
    p_hat = sigmoid(r_hat[winners] - r_hat[losers])
    grad = np.zeros(n_items)
    np.add.at(grad, winners, -(1 - p_hat))
    np.add.at(grad, losers, (1 - p_hat))
    r_hat -= lr * grad / len(winners)

correlation = np.corrcoef(true_reward, r_hat)[0, 1]
print(f"true reward:       {np.round(true_reward, 2)}")
print(f"recovered reward:  {np.round(r_hat, 2)}")
print(f"correlation(true, recovered): {correlation:.3f}")

In [ ]:
plt.figure()
plt.scatter(true_reward, r_hat)
for i, item in enumerate(items):
    plt.annotate(item, (true_reward[i], r_hat[i]), textcoords="offset points", xytext=(5, 5))
plt.xlabel("true hidden reward")
plt.ylabel("recovered reward $\hat r$")
plt.title("Bradley-Terry reward recovery from pairwise comparisons")
plt.tight_layout()
plt.show()

The recovered rewards correlate strongly with the hidden true rewards
and preserve their exact ranking — 400 noisy, pairwise "who won"
comparisons, with no item's absolute quality ever labelled, are enough
to recover a consistent scalar reward. Bradley-Terry's own logistic form
is *exactly* what a real RLHF reward model's loss function is, applied
here to a synthetic ranking problem instead of real human preference
data over language model outputs, purely for speed and reproducibility.

## The RLHF Loop

RLHF assembles three separate pieces into one pipeline:

1. **Collect preference data.** Humans compare pairs of outputs from a
   (usually already fine-tuned, 12a) language model and record which one
   they preferred.
2. **Train a reward model.** Exactly the Bradley-Terry loss above, now
   with $r_\theta$ itself a neural network taking a (prompt, response)
   pair as input, trained on the collected comparisons.
3. **Fine-tune the language model against the reward model, with
   reinforcement learning.** This is the one genuinely *RL* step in the
   pipeline: the language model is treated as a policy $\pi_\theta(y \mid
   x)$ choosing a response $y$ given a prompt $x$, the learned reward
   model $r_\theta(x, y)$ supplies the reward signal, and a KL penalty
   against a frozen reference policy $\pi_{\text{ref}}$ (the model before
   this step) keeps the policy from drifting so far it starts exploiting
   quirks of the reward model rather than genuinely improving:

$$\max_\theta\ \mathbb{E}_{y \sim \pi_\theta(\cdot|x)}\big[r_\theta(x,y)\big] - \beta\, D_{\text{KL}}\big(\pi_\theta(\cdot|x) \,\|\, \pi_{\text{ref}}(\cdot|x)\big).$$

Maximising an expectation over samples drawn from a policy you are
simultaneously trying to update is precisely a policy-gradient
reinforcement learning problem — sampling a response is an *action*
under $\pi_\theta$, and neither the reward model nor the environment
provides a gradient with respect to $\theta$ through that sampling step
directly. **Proximal Policy Optimisation (PPO)** is the specific
algorithm almost every published RLHF pipeline uses to solve exactly
this maximisation step; its clipped surrogate objective and full
derivation are the subject of the RL series' own PPO lesson — this
notebook only needs to know precisely where in the pipeline PPO sits,
not re-derive it: step 3, and step 3 alone, is the reinforcement
learning problem.

## Direct Preference Optimisation

RLHF's three-stage pipeline (reward model, then RL) exists because
maximising expected reward under a policy is normally solved by
sampling and a policy-gradient method. DPO's insight is that for
*this specific* KL-regularised objective, the optimal policy has a
closed form in terms of the reward:

$$\pi^*(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\!\left(\frac{r(x,y)}{\beta}\right) \quad\Longrightarrow\quad r(x,y) = \beta \log \frac{\pi^*(y\mid x)}{\pi_{\text{ref}}(y\mid x)} + \beta \log Z(x).$$

Substituting this expression for $r$ directly into the Bradley-Terry
loss is the entire derivation: the awkward, prompt-dependent
normalising term $\beta \log Z(x)$ appears identically in both the
winning and losing response's reward, so it **cancels exactly** in the
reward *difference* the Bradley-Terry loss actually uses. What remains
is a loss written purely in terms of the policy's own log-probabilities
— the trained policy $\pi_\theta$ and a frozen reference $\pi_{\text{ref}}$
— with **no reward model and no RL loop anywhere in it**:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\log \sigma\!\left(\beta \log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta \log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\right).$$

Gradient descent on $\mathcal{L}_{\text{DPO}}$ directly with ordinary
backpropagation — no sampling, no reward model, no policy rollouts — is
provably optimising for the *same* KL-regularised reward-maximisation
objective RLHF's three-stage pipeline exists to solve.

In [ ]:
# A minimal policy: for each prompt there are exactly two candidate
# responses, so pi_theta(y_w | x) = sigmoid(theta[x]) and
# pi_theta(y_l | x) = sigmoid(-theta[x]) -- theta[x] is the policy's
# entire log-odds in favour of the preferred response for prompt x.
# theta_ref is the frozen reference model (here: genuinely undecided,
# 50/50, before any preference training).
n_prompts = 5
beta = 1.0
theta_ref = np.zeros(n_prompts)


def dpo_loss_and_grad(theta, theta_ref, beta):
    """Closed form for the two-response case (derived in the markdown
    above): z = beta * (theta - theta_ref) exactly, so the DPO loss
    reduces to a plain logistic loss on that quantity."""
    z = beta * (theta - theta_ref)
    loss = -np.log(sigmoid(z) + 1e-12)
    grad = -beta * sigmoid(-z)
    return loss, grad


# Verify the closed form against the literal DPO loss computed from
# log pi_theta and log pi_ref directly, and against torch autograd.
theta_check = np.array([0.7, -0.3, 1.4, 0.0, -1.1])


def literal_dpo_loss(theta, theta_ref, beta):
    log_pi_w = np.log(sigmoid(theta))
    log_pi_l = np.log(sigmoid(-theta))
    log_ref_w = np.log(sigmoid(theta_ref))
    log_ref_l = np.log(sigmoid(-theta_ref))
    z = beta * (log_pi_w - log_ref_w) - beta * (log_pi_l - log_ref_l)
    return -np.log(sigmoid(z) + 1e-12)


loss_closed, grad_closed = dpo_loss_and_grad(theta_check, theta_ref, beta)
loss_literal = literal_dpo_loss(theta_check, theta_ref, beta)
print(f"max abs diff, closed-form vs literal DPO loss: {np.abs(loss_closed - loss_literal).max():.2e}")

theta_t = torch.tensor(theta_check, requires_grad=True)
theta_ref_t = torch.tensor(theta_ref)
log_pi_w_t = torch.log(torch.sigmoid(theta_t))
log_pi_l_t = torch.log(torch.sigmoid(-theta_t))
log_ref_w_t = torch.log(torch.sigmoid(theta_ref_t))
log_ref_l_t = torch.log(torch.sigmoid(-theta_ref_t))
z_t = beta * (log_pi_w_t - log_ref_w_t) - beta * (log_pi_l_t - log_ref_l_t)
loss_t = -torch.log(torch.sigmoid(z_t) + 1e-12)
loss_t.sum().backward()
print(f"max abs diff, closed-form grad vs torch autograd: {np.abs(grad_closed - theta_t.grad.numpy()).max():.2e}")

In [ ]:
# Train the policy on nothing but the five preference pairs.
theta = np.zeros(n_prompts)
lr, n_epochs = 0.5, 200
loss_history = []
for epoch in range(n_epochs):
    loss, grad = dpo_loss_and_grad(theta, theta_ref, beta)
    theta -= lr * grad
    loss_history.append(loss.mean())

pi_before = sigmoid(theta_ref)
pi_after = sigmoid(theta)
print(f"P(preferred response) before DPO training: {np.round(pi_before, 2)}")
print(f"P(preferred response) after DPO training:  {np.round(pi_after, 2)}")

In [ ]:
plt.figure()
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("mean DPO loss")
plt.title("Training directly on preference pairs, no reward model")
plt.tight_layout()
plt.show()

Every one of the five prompts starts at exactly 50/50 (the reference
policy is genuinely undecided) and, after training on nothing but the
preference labels themselves, the policy assigns the preferred response
a much higher probability — with no reward model ever instantiated, no
sampling step, and no RL algorithm anywhere in the training loop. The
closed-form loss derived above matched both a literal recomputation from
$\log \pi_\theta$ and $\log \pi_{\text{ref}}$ and PyTorch's autograd
gradient to floating-point precision.

## Key Takeaways

- **Preference comparisons, not hand-written scores, are what
  alignment data actually is** — humans are far more reliable at "which
  is better" than at absolute scoring, and a hand-written reward is
  exactly what gets exploited when it is even slightly wrong.
- **The Bradley-Terry model turns comparisons into a trainable scalar
  reward** via a logistic loss on reward differences — recovered here
  from 400 synthetic comparisons with strong correlation to the hidden
  true reward, and used identically in real RLHF reward models.
- **RLHF is three stages, and only one of them is reinforcement
  learning**: collect preferences, fit a reward model (Bradley-Terry),
  then use RL (PPO, from the RL series) to maximise that reward under a
  KL penalty against drifting away from the reference model.
- **DPO substitutes the KL-regularised objective's closed-form optimal
  policy back into the Bradley-Terry loss**, cancelling the
  prompt-dependent normaliser exactly and leaving a loss purely in terms
  of policy log-probabilities — verified here against both a literal
  recomputation and PyTorch autograd — that trains directly on
  preference pairs with no reward model and no RL loop at all.